# 03 — Cross-polarized BIOMASS TomoSAR


## Biomass Data Inspection

### Data Loading & Analysis

In [ ]:
from biomass_tomosar.paths import biomass_stack_dir

DATA_DIR = biomass_stack_dir()
MASTER_IDX = 3  # Fourth acquisition (Python uses zero-based indexing)

assert DATA_DIR.is_dir(), f"BIOMASS stack folder not found: {DATA_DIR}"

products = sorted(p for p in DATA_DIR.glob("BIO*") if p.is_dir())
print(f"Data directory: {DATA_DIR}")
print(f"Product folders found: {len(products)}")
assert len(products) == 7, "Expected exactly seven BIOMASS acquisitions. Check the folder path."

for i, product in enumerate(products):
    stem = product.name[:70].lower()  # Naming rule in the original loader
    amplitude = product / "measurement" / f"{stem}_i_abs.tiff"
    phase = product / "measurement" / f"{stem}_i_phase.tiff"
    marker = " <-- REFERENCE (4th)" if i == MASTER_IDX else ""
    print(f"{i + 1}. {product.name}{marker}")
    print(f"   Amplitude TIFF: {amplitude.exists()} | Phase TIFF: {phase.exists()}")

assert all(
    (p / "measurement" / f"{p.name[:70].lower()}_i_abs.tiff").is_file()
    and (p / "measurement" / f"{p.name[:70].lower()}_i_phase.tiff").is_file()
    for p in products
), "At least one expected measurement TIFF is missing."
print("\nStep 1 complete: all expected measurement TIFF paths exist.")

In [ ]:
from osgeo import gdal

gdal.UseExceptions()

for i, product in enumerate(products):
    stem = product.name[:70].lower()
    measurement = product / "measurement"

    amp_path = measurement / f"{stem}_i_abs.tiff"
    phase_path = measurement / f"{stem}_i_phase.tiff"

    amp_ds = gdal.Open(str(amp_path), gdal.GA_ReadOnly)
    phase_ds = gdal.Open(str(phase_path), gdal.GA_ReadOnly)

    # Check amplitude and phase have identical dimensions
    assert (
        amp_ds.RasterXSize == phase_ds.RasterXSize
        and amp_ds.RasterYSize == phase_ds.RasterYSize
        and amp_ds.RasterCount == phase_ds.RasterCount
    ), f"TIFF mismatch in acquisition {i+1}"

    print(f"Acquisition {i+1}")
    print(f"  Shape (az, rg): {amp_ds.RasterYSize, amp_ds.RasterXSize}")
    print(f"  Number of bands: {amp_ds.RasterCount}")
    print(f"  Polarizations: {amp_ds.GetMetadataItem('PolarisationsSequence')}")

    if i == 0:
        print("  Band details:")
        for b in range(1, amp_ds.RasterCount + 1):
            band = amp_ds.GetRasterBand(b)
            print(
                f"    Band {b}: "
                f"{band.GetMetadataItem('POLARIMETRIC_INTERP') or band.GetDescription()}"
            )

    amp_ds = None
    phase_ds = None

In [ ]:
from biomass_tomosar.bm_data_loader import bmStack

# Initialize stack metadata and NetCDF LUT access
base_dirs = [str(p) for p in products]
stack = bmStack(base_dirs)

print(f"Number of acquisitions: {stack.Ntracks}")
print(f"Image dimensions: {stack.nli} × {stack.nsa}")
print(f"Radar wavelength: {stack.lamb:.4f} m")
print(f"Reference acquisition index: {MASTER_IDX}")

# Inspect available phase screens for every acquisition
for i, lut in enumerate(stack.luts):
    nc = lut.get().nc

    flat = nc.groups["coregistration"].variables[
        "flatteningPhaseScreen"
    ]

    skp = nc.groups["skpPhaseCalibration"].variables[
        "skpCalibrationPhaseScreen"
    ]

    quality = nc.groups["skpPhaseCalibration"].variables[
        "skpCalibrationPhaseScreenQuality"
    ]

    print(f"\nAcquisition {i+1}")
    print(f"  Flattening phase: {flat.shape} [{flat.units}]")
    print(f"  SKP phase:        {skp.shape} [{skp.units}]")
    print(f"  SKP quality:      {quality.shape}")


In [ ]:
import numpy as np

# Read metadata from the reference acquisition
annot = stack.annot_coreg[MASTER_IDX]

C = 299_792_458  # Speed of light [m/s]


def get_parameter(name, parent=None):
    """Extract a numerical parameter from BIOMASS XML annotations."""
    if parent:
        xpath = (
            f"//*[local-name()='{parent}']"
            f"//*[local-name()='{name}']"
        )
    else:
        xpath = f"//*[local-name()='{name}']"

    nodes = annot.xpath(xpath)

    if not nodes:
        return None

    text = "".join(nodes[0].itertext()).strip()

    try:
        return float(text)
    except ValueError:
        return text


def show(label, value, unit="", precision=3):
    if value is None:
        print(f"{label:35s}: Not available")
    elif isinstance(value, (float, int, np.number)):
        print(f"{label:35s}: {value:,.{precision}f} {unit}")
    else:
        print(f"{label:35s}: {value} {unit}")


# ---------------------------------------------------------
# Instrument parameters
# ---------------------------------------------------------
fc = get_parameter("radarCarrierFrequency", "instrumentParameters")
wavelength = stack.lamb

# ---------------------------------------------------------
# SAR sampling parameters
# ---------------------------------------------------------
rg_spacing = get_parameter("rangePixelSpacing", "sarImage")
az_spacing = get_parameter("azimuthPixelSpacing", "sarImage")

rg_dt = get_parameter("rangeTimeInterval", "sarImage")
az_dt = get_parameter("azimuthTimeInterval", "sarImage")

# ---------------------------------------------------------
# Processing bandwidths
# ---------------------------------------------------------
rg_bw = get_parameter(
    "processingBandwidth", "rangeProcessingParameters"
)
az_bw = get_parameter(
    "processingBandwidth", "azimuthProcessingParameters"
)

rg_total_bw = get_parameter(
    "totalBandwidth", "rangeProcessingParameters"
)
az_total_bw = get_parameter(
    "totalBandwidth", "azimuthProcessingParameters"
)

# ---------------------------------------------------------
# Theoretical / approximate spatial resolutions
# ---------------------------------------------------------
rg_resolution = (
    C / (2 * rg_bw) if isinstance(rg_bw, (int, float)) and rg_bw > 0
    else None
)

# Approximate along-track velocity inferred from image sampling
az_velocity = (
    az_spacing / az_dt
    if isinstance(az_spacing, (int, float))
    and isinstance(az_dt, (int, float))
    and az_dt > 0
    else None
)

az_resolution = (
    az_velocity / az_bw
    if az_velocity is not None
    and isinstance(az_bw, (int, float))
    and az_bw > 0
    else None
)

# ---------------------------------------------------------
# Display
# ---------------------------------------------------------
print("=" * 57)
print("BIOMASS — RADAR & SAR IMAGE PARAMETERS")
print("=" * 57)

print("\nINSTRUMENT")
show("Carrier frequency", fc / 1e6 if isinstance(fc, (int, float)) else None, "MHz")
show("Wavelength", wavelength, "m", 4)

print("\nIMAGE GEOMETRY")
show("Azimuth lines", stack.nli, "pixels", 0)
show("Range samples", stack.nsa, "pixels", 0)
show("Azimuth pixel spacing", az_spacing, "m")
show("Slant-range pixel spacing", rg_spacing, "m")

print("\nSAMPLING")
show("Range sampling frequency", 1 / rg_dt / 1e6 if isinstance(rg_dt, (int, float)) and rg_dt > 0 else None, "MHz")
show("Output azimuth line rate", 1 / az_dt if isinstance(az_dt, (int, float)) and az_dt > 0 else None, "Hz")

print("\nPROCESSING BANDWIDTH")
show("Range available bandwidth", rg_total_bw / 1e6 if isinstance(rg_total_bw, (int, float)) else None, "MHz")
show("Range processing bandwidth", rg_bw / 1e6 if isinstance(rg_bw, (int, float)) else None, "MHz")
show("Azimuth available bandwidth", az_total_bw, "Hz")
show("Azimuth processing bandwidth", az_bw, "Hz")

print("\nRESOLUTION ESTIMATES")
show("Slant-range resolution", rg_resolution, "m")
show("Azimuth resolution (approx.)", az_resolution, "m")

print("=" * 57)

### Memory Usage

In [ ]:
import numpy as np

def fmt_size(nbytes):
    return f"{nbytes / 2**20:.1f} MiB ({nbytes / 2**30:.2f} GiB)"

# Image dimensions
n_acq = len(stack.luts)
n_pixels = stack.nli * stack.nsa

# Uncompressed LUT storage
lut_bytes = 0

lut_variables = [
    ("coregistration", "flatteningPhaseScreen"),
    ("skpPhaseCalibration", "skpCalibrationPhaseScreen"),
    ("skpPhaseCalibration", "skpCalibrationPhaseScreenQuality"),
]

for lut in stack.luts:
    nc = lut.get().nc

    for group, name in lut_variables:
        var = nc.groups[group].variables[name]
        lut_bytes += np.prod(var.shape) * np.dtype(var.dtype).itemsize

# Expected SLC storage (complex64 = 8 bytes per pixel)
bytes_per_slc = n_pixels * np.dtype(np.complex64).itemsize

print("\n--- Estimated memory requirements ---")
print(f"3 LUTs × {n_acq} acquisitions: {fmt_size(lut_bytes)}")
print(f"Full SLC stack (4 pols):     {fmt_size(bytes_per_slc * n_acq * 4)}")
print(f"HV + VH stack (2 pols):      {fmt_size(bytes_per_slc * n_acq * 2)}")
print(f"Combined cross-pol stack:   {fmt_size(bytes_per_slc * n_acq)}")

# Actual memory usage of current Python kernel
print("\n--- Current Python process memory ---")

with open("/proc/self/status") as f:
    for line in f:
        if line.startswith(("VmRSS:", "VmSwap:")):
            key, value, _ = line.split()
            print(f"{key}: {fmt_size(int(value) * 1024)}")

### Inspect Correction Phases

#### Image 1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Select acquisition (0-based index)
ACQ_IDX = 0  # First secondary acquisition
lut = stack.luts[ACQ_IDX]

# Read LUTs at their native resolution
flat = lut.get(
    "coregistration", "flatteningPhaseScreen"
).nc[:]

skp = lut.get(
    "skpPhaseCalibration", "skpCalibrationPhaseScreen"
).nc[:]

quality = lut.get(
    "skpPhaseCalibration", "skpCalibrationPhaseScreenQuality"
).nc[:]

# Preserve NetCDF no-data masks and mask NaN/Inf
flat = np.ma.masked_invalid(flat)
skp = np.ma.masked_invalid(skp)
quality = np.ma.masked_invalid(quality)

print(f"Acquisition: {ACQ_IDX + 1}")
print(f"Flattening LUT: {flat.shape}")
print(f"SKP LUT:        {skp.shape}")
print(f"Quality LUT:    {quality.shape}")

for name, array in [("Flattening", flat), ("SKP", skp), ("Quality", quality)]:
    values = array.compressed()
    if values.size:
        print(
            f"{name}: min={values.min():.3f}, "
            f"max={values.max():.3f}, "
            f"mean={values.mean():.3f}, "
            f"valid={values.size / array.size:.1%}"
        )

# Wrap phases to [-pi, pi] for visualization only
def wrap_phase(phase):
    return np.ma.arctan2(np.sin(phase), np.cos(phase))

fig, axes = plt.subplots(1, 3, figsize=(16, 6))

plots = [
    (wrap_phase(flat), "Flattening phase", "twilight", -np.pi, np.pi),
    (wrap_phase(skp), "SKP calibration phase", "twilight", -np.pi, np.pi),
    (quality, "SKP estimation quality", "viridis", 0, 1),
]

for ax, (data, title, cmap, vmin, vmax) in zip(axes, plots):
    im = ax.imshow(
        data,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        aspect="auto",
        interpolation="nearest"
    )

    ax.set_title(title)
    ax.set_xlabel("Slant-range LUT index")
    ax.set_ylabel("Azimuth LUT index")

    fig.colorbar(
        im,
        ax=ax,
        label="Quality" if title.endswith("quality") else "Phase [rad]"
    )

fig.suptitle(f"BIOMASS — Acquisition {ACQ_IDX + 1}")
plt.tight_layout()
plt.show()

#### Image 4 : Master


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Select acquisition (0-based index)
ACQ_IDX = 3  # First secondary acquisition
lut = stack.luts[ACQ_IDX]

# Read LUTs at their native resolution
flat = lut.get(
    "coregistration", "flatteningPhaseScreen"
).nc[:]

skp = lut.get(
    "skpPhaseCalibration", "skpCalibrationPhaseScreen"
).nc[:]

quality = lut.get(
    "skpPhaseCalibration", "skpCalibrationPhaseScreenQuality"
).nc[:]

# Preserve NetCDF no-data masks and mask NaN/Inf
flat = np.ma.masked_invalid(flat)
skp = np.ma.masked_invalid(skp)
quality = np.ma.masked_invalid(quality)

print(f"Acquisition: {ACQ_IDX + 1}")
print(f"Flattening LUT: {flat.shape}")
print(f"SKP LUT:        {skp.shape}")
print(f"Quality LUT:    {quality.shape}")

for name, array in [("Flattening", flat), ("SKP", skp), ("Quality", quality)]:
    values = array.compressed()
    if values.size:
        print(
            f"{name}: min={values.min():.3f}, "
            f"max={values.max():.3f}, "
            f"mean={values.mean():.3f}, "
            f"valid={values.size / array.size:.1%}"
        )

# Wrap phases to [-pi, pi] for visualization only
def wrap_phase(phase):
    return np.ma.arctan2(np.sin(phase), np.cos(phase))

fig, axes = plt.subplots(1, 3, figsize=(16, 6))

plots = [
    (wrap_phase(flat), "Flattening phase", "twilight", -np.pi, np.pi),
    (wrap_phase(skp), "SKP calibration phase", "twilight", -np.pi, np.pi),
    (quality, "SKP estimation quality", "viridis", 0, 1),
]

for ax, (data, title, cmap, vmin, vmax) in zip(axes, plots):
    im = ax.imshow(
        data,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        aspect="auto",
        interpolation="nearest"
    )

    ax.set_title(title)
    ax.set_xlabel("Slant-range LUT index")
    ax.set_ylabel("Azimuth LUT index")

    fig.colorbar(
        im,
        ax=ax,
        label="Quality" if title.endswith("quality") else "Phase [rad]"
    )

fig.suptitle(f"BIOMASS — Acquisition {ACQ_IDX + 1}")
plt.tight_layout()
plt.show()

## Load SLC's and Phase Correction : Full Images

In [ ]:
import numpy as np
from osgeo import gdal
from tqdm.auto import tqdm

# Final stack: (acquisition, azimuth, range)
xp_stack = np.empty(
    (stack.Ntracks, stack.nli, stack.nsa),
    dtype=np.complex64
)

def read_slc(amp_ds, phase_ds, band):
    amplitude = amp_ds.GetRasterBand(band).ReadAsArray()
    phase = phase_ds.GetRasterBand(band).ReadAsArray()

    invalid = (
        (amplitude == stack.noDataValue)
        | (phase == stack.noDataValue)
        | ~np.isfinite(amplitude)
        | ~np.isfinite(phase)
    )

    slc = (amplitude * np.exp(1j * phase)).astype(np.complex64)
    slc[invalid] = 0

    return slc


for i, (product, lut) in enumerate(
    tqdm(zip(products, stack.luts), total=stack.Ntracks)
):
    stem = product.name[:70].lower()

    amp_path = product / "measurement" / f"{stem}_i_abs.tiff"
    phase_path = product / "measurement" / f"{stem}_i_phase.tiff"

    amp_ds = gdal.Open(str(amp_path))
    phase_ds = gdal.Open(str(phase_path))

    # 1. Load HV and VH
    hv = read_slc(amp_ds, phase_ds, 2)
    vh = read_slc(amp_ds, phase_ds, 3)

    # 2. Interpolate phase corrections to full SLC resolution
    flat = lut.get(
        "coregistration", "flatteningPhaseScreen"
    ).upsampled

    skp = lut.get(
        "skpPhaseCalibration", "skpCalibrationPhaseScreen"
    ).upsampled

    assert flat.shape == hv.shape
    assert skp.shape == hv.shape

    # 3. Compute total phase correction
    total_phase = flat + skp
    invalid_phase = (
        ~np.isfinite(total_phase)
        | ~np.isfinite(flat)
        | ~np.isfinite(skp)
        | (flat == -9999)
        | (skp == -9999)
    )
    total_phase[invalid_phase] = 0

    correction = np.exp(1j * total_phase).astype(np.complex64)

    # 4. Apply correction to both polarizations
    hv *= correction
    vh *= correction

    # 5. Combine HV and VH (normalized cross-pol)
    xp_stack[i] = (hv + vh) / np.sqrt(2)
    xp_stack[i][invalid_phase] = 0

    print(f"Acquisition {i+1}: processed")


print("\n--- Final corrected cross-pol stack ---")
print("Shape:", xp_stack.shape)
print("Dtype:", xp_stack.dtype)
print(f"Memory: {xp_stack.nbytes / 2**30:.2f} GiB")

### Save Stack and metadata

In [ ]:
import importlib
import biomass_tomosar.paths as paths

importlib.reload(paths)

output_dir = paths.processed_data_dir()
output_dir.mkdir(parents=True, exist_ok=True)

print("Processed data directory:", output_dir)

In [ ]:
import numpy as np
import json
from biomass_tomosar.paths import processed_data_dir

# 1. Output directory from config/local.toml
output_dir = processed_data_dir()
output_dir.mkdir(parents=True, exist_ok=True)

slc_file = output_dir / "xp_stack.npy"
metadata_file = output_dir / "xp_stack_metadata.json"

# 2. Save the complete corrected SLC stack
np.save(slc_file, xp_stack, allow_pickle=False)

# 3. Extract grid parameters from the reference acquisition XML
annot = stack.annot_coreg[MASTER_IDX]

def xml_value(name):
    nodes = annot.xpath(
        f"//*[local-name()='sarImage']/*[local-name()='{name}']"
    )
    return nodes[0].text.strip() if nodes and nodes[0].text else None

# 4. Save metadata
metadata = {
    "shape": list(xp_stack.shape),
    "dtype": str(xp_stack.dtype),
    "axis_order": ["acquisition", "azimuth", "range"],
    "polarization": "(HV + VH) / sqrt(2)",
    "phase_corrections": ["flattening", "SKP"],
    "reference_acquisition_index": MASTER_IDX,
    "acquisitions": [p.name for p in products],
    "wavelength_m": float(stack.lamb),
    "grid": {
        "number_of_azimuth_lines": int(stack.nli),
        "number_of_range_samples": int(stack.nsa),
        "azimuth_pixel_spacing_m": xml_value("azimuthPixelSpacing"),
        "range_pixel_spacing_m": xml_value("rangePixelSpacing"),
        "azimuth_time_interval_s": xml_value("azimuthTimeInterval"),
        "range_time_interval_s": xml_value("rangeTimeInterval"),
        "projection": xml_value("projection"),
    },
}

with metadata_file.open("w") as f:
    json.dump(metadata, f, indent=4)

# 5. Verify saved files
print(f"SLC saved: {slc_file}")
print(f"Metadata saved: {metadata_file}")
print(f"Stack size: {slc_file.stat().st_size / 2**30:.2f} GiB")

### Plot SLC's

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Display settings
STEP = 8  # Downsampling for visualization only
VMIN, VMAX = -35, 5  # Common dB scale across acquisitions

fig, axes = plt.subplots(
    2, 4,
    figsize=(18, 12),
    constrained_layout=True
)

for i, ax in enumerate(axes.flat):

    if i >= xp_stack.shape[0]:
        ax.axis("off")
        continue

    # Read a downsampled view to avoid large plotting arrays
    slc = xp_stack[i, ::STEP, ::STEP]

    amplitude_db = 20 * np.log10(
        np.maximum(np.abs(slc), 1e-8)
    )

    im = ax.imshow(
        amplitude_db,
        cmap="gray",
        vmin=VMIN,
        vmax=VMAX,
        aspect="auto"
    )

    title = f"Acquisition {i+1}"
    if i == MASTER_IDX:
        title += " — REFERENCE"

    ax.set_title(title, fontweight="bold" if i == MASTER_IDX else "normal")
    ax.set_xlabel("Range")
    ax.set_ylabel("Azimuth")

fig.colorbar(
    im,
    ax=axes,
    label="Cross-pol amplitude [dB]",
    shrink=0.65,
    pad=0.02
)

plt.show()

In [ ]:
import gc
import os

# Remove temporary arrays, keeping xp_stack
for name in [
    "hv", "vh", "flat", "skp",
    "total_phase", "correction",
    "invalid_phase"
]:
    globals().pop(name, None)

gc.collect()

print(f"Kernel PID: {os.getpid()}")
print(f"XP stack: {xp_stack.nbytes / 2**30:.2f} GiB")

with open("/proc/self/status") as f:
    for line in f:
        if line.startswith(("VmRSS:", "VmSwap:")):
            print(line.strip())

## Load LiDAR Profiles


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from biomass_tomosar.paths import gedi_data_dir

# ---------------------------------------------------------
# 1. Locate GEDI data using config/local.toml
# ---------------------------------------------------------
gedi_dir = gedi_data_dir()

# ---------------------------------------------------------
# 2. Load interpolated forest height and grid parameters
# ---------------------------------------------------------
gedi_height = np.load(gedi_dir / "gedi_forestHeight.npy")

grid = pd.read_csv(gedi_dir / "gedi_forestHeight.grd").iloc[0]

az0, naz, daz = grid["az0"], int(grid["naz"]), grid["daz"]
rg0, nrg, drg = grid["rg0"], int(grid["nrg"]), grid["drg"]

gedi_az = az0 + daz * np.arange(naz)
gedi_rg = rg0 + drg * np.arange(nrg)

assert gedi_height.shape == (naz, nrg)

# ---------------------------------------------------------
# 3. Load raw GEDI measurements
# ---------------------------------------------------------
gedi_points = pd.read_csv(
    gedi_dir / "gedi.csv",
    usecols=["rg", "az", "rh_98", "rh_0"],
)

gedi_points["forest_height"] = (
    gedi_points["rh_98"] - gedi_points["rh_0"]
)

# ---------------------------------------------------------
# 4. Display metadata
# ---------------------------------------------------------
print("GEDI FOREST HEIGHT")
print(f"Data directory: {gedi_dir}")
print(f"Raw GEDI shots: {len(gedi_points):,}")
print(f"Grid shape (az, rg): {gedi_height.shape}")
print(f"Grid spacing: {daz:.0f} m azimuth × {drg:.0f} m range")
print(f"Range extent: {gedi_rg[0]:.1f} to {gedi_rg[-1]:.1f} m")
print(f"Azimuth extent: {gedi_az[0]:.1f} to {gedi_az[-1]:.1f} m")
print(
    f"Interpolated height: "
    f"{np.nanmin(gedi_height):.1f} to "
    f"{np.nanmax(gedi_height):.1f} m"
)

# ---------------------------------------------------------
# 5. Visualize raw GEDI points and interpolated forest height
# ---------------------------------------------------------
fig, axes = plt.subplots(
    1, 2,
    figsize=(11, 11),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)

vmin, vmax = 0, 50  # Forest height [m]

# Raw GEDI tracks
axes[0].scatter(
    gedi_points["rg"],
    gedi_points["az"],
    c=gedi_points["forest_height"],
    s=2,
    cmap="viridis",
    vmin=vmin,
    vmax=vmax,
    rasterized=True,
)

axes[0].set_title("Raw GEDI LiDAR tracks")
axes[0].set_ylabel("Azimuth [m]")

# Interpolated 60 m grid
im = axes[1].imshow(
    gedi_height,
    extent=[
        gedi_rg[0] - drg / 2,
        gedi_rg[-1] + drg / 2,
        gedi_az[0] - daz / 2,
        gedi_az[-1] + daz / 2,
    ],
    origin="lower",
    cmap="viridis",
    vmin=vmin,
    vmax=vmax,
    interpolation="nearest",
)

axes[1].set_title("Interpolated GEDI forest height (60 m)")

# Common axis settings
for ax in axes:
    ax.set_xlabel("Range [m]")
    ax.set_aspect("equal")
    ax.set_xlim(gedi_rg[0], gedi_rg[-1])

    # Reverse azimuth axis: smallest azimuth at the top
    ax.set_ylim(gedi_az[-1], gedi_az[0])

fig.colorbar(
    im,
    ax=axes,
    label="Forest height [m]",
    shrink=0.65,
)

plt.show()

## Load Stack from Saved Data

In [ ]:
import numpy as np
import json

from biomass_tomosar.paths import processed_data_dir

# Paths from config/local.toml
data_dir = processed_data_dir()

slc_file = data_dir / "xp_stack.npy"
metadata_file = data_dir / "xp_stack_metadata.json"

# Load the SLC stack without copying everything into RAM
xp_stack = np.load(slc_file, mmap_mode="r")

# Load metadata
with metadata_file.open("r") as f:
    xp_metadata = json.load(f)

# Useful parameters
MASTER_IDX = xp_metadata["reference_acquisition_index"]
wavelength = xp_metadata["wavelength_m"]
acquisitions = xp_metadata["acquisitions"]
grid = xp_metadata["grid"]

print("XP stack loaded")
print(f"Shape: {xp_stack.shape}")
print(f"Dtype: {xp_stack.dtype}")
print(f"Polarization: {xp_metadata['polarization']}")
print(f"Reference acquisition: {MASTER_IDX + 1}")
print(f"Wavelength: {wavelength:.4f} m")
print(f"Memory-mapped file size: {xp_stack.nbytes / 2**30:.2f} GiB")
print("\nGrid parameters:")
for key, value in grid.items():
    print(f"  {key}: {value}")

## TomoSAR Processing

### Import Kz

In [ ]:
import numpy as np

from biomass_tomosar.paths import biomass_stack_dir
from biomass_tomosar.bm_data_loader import bmStack

# ---------------------------------------------------------
# 1. Locate the BIOMASS acquisitions
# ---------------------------------------------------------
data_dir = biomass_stack_dir()

products = sorted(
    p for p in data_dir.iterdir()
    if p.is_dir() and p.name.startswith("BIO_S2_STA")
)

assert len(products) == 7, f"Expected 7 acquisitions, found {len(products)}"

# ---------------------------------------------------------
# 2. Initialize stack (metadata and LUTs only)
# ---------------------------------------------------------
stack = bmStack([str(p) for p in products])

MASTER_IDX = 3

# ---------------------------------------------------------
# 3. Load vertical wavenumbers at native LUT resolution
# ---------------------------------------------------------
kz_stack = np.stack([
    np.asarray(
        np.ma.filled(
            lut.get("coregistration", "waveNumbers").nc[:],
            np.nan
        ),
        dtype=np.float32
    )
    for lut in stack.luts
])

# ---------------------------------------------------------
# 4. Display information
# ---------------------------------------------------------
print("Vertical wavenumbers loaded")
print("Shape:", kz_stack.shape)
print("Dtype:", kz_stack.dtype)
print(f"Memory: {kz_stack.nbytes / 2**20:.1f} MiB")

for i, kz in enumerate(kz_stack):
    valid = kz[np.isfinite(kz) & (kz != -9999)]

    if valid.size:
        print(
            f"Acquisition {i + 1}: "
            f"kz = {valid.min():.5f} to {valid.max():.5f} rad/m"
        )
    else:
        print(f"Acquisition {i + 1}: no valid kz values")

### Compute Kz Mean for each range line

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# 1. Read the native LUT range coordinates
# ---------------------------------------------------------
kz_var = stack.luts[0].get("coregistration", "waveNumbers").nc

# Coordinate variables are stored in the NetCDF root group
root = kz_var.group()
while root.parent is not None:
    root = root.parent

range_dim = kz_var.dimensions[1]
range_time = np.asarray(root.variables[range_dim][:])

# Two-way slant-range time [s] -> slant-range distance [km]
C = 299_792_458
range_km = C * range_time / 2 / 1000

# ---------------------------------------------------------
# 2. Average kz along azimuth
# ---------------------------------------------------------
kz_valid = np.where(
    np.isfinite(kz_stack) & (kz_stack != -9999),
    kz_stack,
    np.nan,
)

# Shape: (7 acquisitions, 784 range samples)
kz_mean = np.nanmean(kz_valid, axis=1)

assert kz_mean.shape[1] == len(range_km)

# ---------------------------------------------------------
# 3. Plot
# ---------------------------------------------------------
fig, ax = plt.subplots(figsize=(11, 5))

for i in range(kz_mean.shape[0]):
    is_master = (i == MASTER_IDX)

    ax.plot(
        range_km,
        kz_mean[i],
        label=f"Acquisition {i + 1}" + (" (reference)" if is_master else ""),
        linewidth=2 if is_master else 1.5,
        linestyle="--" if is_master else "-",
    )

ax.axhline(0, color="black", linewidth=0.8, alpha=0.5)

ax.set_xlabel("Slant range [km]")
ax.set_ylabel(r"Mean vertical wavenumber $k_z$ [rad/m]")
ax.set_title(r"Mean $k_z$ along azimuth vs. slant range")

ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

### Vizualize Height Resolution for each range


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# 1. Find kz extrema across acquisitions at each range
# ---------------------------------------------------------
# kz_mean shape: (7 acquisitions, 784 range samples)

kz_min = np.nanmin(kz_mean, axis=0)
kz_max = np.nanmax(kz_mean, axis=0)

# Total vertical wavenumber span [rad/m]
delta_kz = kz_max - kz_min

# ---------------------------------------------------------
# 2. Compute nominal vertical resolution [m]
# ---------------------------------------------------------
delta_z = np.full_like(delta_kz, np.nan)

valid = np.isfinite(delta_kz) & (delta_kz > 0)

delta_z[valid] = 2 * np.pi / delta_kz[valid]

# ---------------------------------------------------------
# 3. Plot
# ---------------------------------------------------------
fig, ax = plt.subplots(figsize=(11, 5))

ax.plot(
    range_km,
    delta_z,
    color="tab:blue",
    linewidth=2,
)

ax.set_xlabel("Slant range [km]")
ax.set_ylabel(r"Vertical resolution $\delta_z$ [m]")
ax.set_title("BIOMASS TomoSAR — Nominal Vertical Resolution")

ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# 4. Display statistics
# ---------------------------------------------------------
print(f"Minimum resolution: {np.nanmin(delta_z):.2f} m")
print(f"Maximum resolution: {np.nanmax(delta_z):.2f} m")
print(f"Mean resolution:    {np.nanmean(delta_z):.2f} m")

### Vizualize height Ambiguity for each range

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# 1. Sort kz values across acquisitions at each range
# ---------------------------------------------------------
# kz_mean shape: (7 acquisitions, 784 range samples)

kz_sorted = np.sort(kz_mean, axis=0)

# Differences between consecutive sorted kz values
kz_diffs = np.diff(kz_sorted, axis=0)

# Minimum strictly positive kz spacing
# (ignore missing values and duplicate kz values)
valid_diffs = np.where(
    np.isfinite(kz_diffs) & (kz_diffs > 0),
    kz_diffs,
    np.inf
)

delta_kz_min = np.min(valid_diffs, axis=0)
delta_kz_min[~np.isfinite(delta_kz_min)] = np.nan

# ---------------------------------------------------------
# 2. Calculate ambiguity height
# ---------------------------------------------------------
h_ambiguity = 2 * np.pi / delta_kz_min

# ---------------------------------------------------------
# 3. Plot
# ---------------------------------------------------------
fig, axes = plt.subplots(
    2, 1,
    figsize=(11, 8),
    sharex=True,
    constrained_layout=True
)

# Minimum kz spacing
axes[0].plot(range_km, delta_kz_min, linewidth=2)
axes[0].set_ylabel(r"Minimum $\Delta k_z$ [rad/m]")
axes[0].set_title("Minimum Vertical Wavenumber Spacing")
axes[0].grid(True, alpha=0.3)

# Ambiguity height
axes[1].plot(range_km, h_ambiguity, linewidth=2)
axes[1].set_xlabel("Slant range [km]")
axes[1].set_ylabel("Ambiguity height [m]")
axes[1].set_title("TomoSAR Vertical Ambiguity")
axes[1].grid(True, alpha=0.3)

plt.show()

# ---------------------------------------------------------
# 4. Statistics
# ---------------------------------------------------------
print(f"Minimum ambiguity height: {np.nanmin(h_ambiguity):.2f} m")
print(f"Maximum ambiguity height: {np.nanmax(h_ambiguity):.2f} m")
print(f"Mean ambiguity height:    {np.nanmean(h_ambiguity):.2f} m")

### PSF 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# 1. Height grid
# ---------------------------------------------------------
Z_MAX = 200   # Maximum relative height [m]
DZ = 0.5      # Height sampling [m]

z = np.arange(-Z_MAX, Z_MAX + DZ, DZ)

n_acq, n_rg = kz_mean.shape

# ---------------------------------------------------------
# 2. Compute the vertical PSF at every range
# ---------------------------------------------------------
# Result shape: (range, height)
psf_sum = np.zeros((n_rg, len(z)), dtype=np.complex64)

n_valid = np.sum(np.isfinite(kz_mean), axis=0)

for i in range(n_acq):
    kz = kz_mean[i]

    valid = np.isfinite(kz)

    psf_sum[valid] += np.exp(
        1j * kz[valid, None] * z[None, :]
    ).astype(np.complex64)

# Normalize by the number of valid acquisitions
psf = np.abs(
    psf_sum / np.maximum(n_valid[:, None], 1)
)**2

psf[n_valid < 2] = np.nan

# Convert power to dB
psf_db = 10 * np.log10(np.maximum(psf, 1e-10))

# ---------------------------------------------------------
# 3. Plot PSF over range and height
# ---------------------------------------------------------
fig, axes = plt.subplots(
    2, 1,
    figsize=(12, 9),
    constrained_layout=True
)

im = axes[0].imshow(
    psf_db.T,
    extent=[range_km[0], range_km[-1], z[0], z[-1]],
    origin="lower",
    aspect="auto",
    cmap="viridis",
    vmin=-40,
    vmax=0,
)

axes[0].axhline(0, color="white", linestyle="--", linewidth=0.8)
axes[0].set_xlabel("Slant range [km]")
axes[0].set_ylabel("Relative height [m]")
axes[0].set_title("Vertical TomoSAR PSF across range")

fig.colorbar(im, ax=axes[0], label="Normalized PSF [dB]")

# ---------------------------------------------------------
# 4. Plot PSF at the center of the range swath
# ---------------------------------------------------------
RG_IDX = n_rg // 2

axes[1].plot(
    z,
    psf_db[RG_IDX],
    linewidth=1.5,
    label=f"Range = {range_km[RG_IDX]:.1f} km"
)

axes[1].axvline(0, color="black", linestyle="--", alpha=0.5)
axes[1].set_xlabel("Relative height [m]")
axes[1].set_ylabel("Normalized PSF [dB]")
axes[1].set_title("Vertical PSF at center range")
axes[1].set_ylim(-40, 2)
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.show()

print("PSF shape:", psf.shape)
print("Height interval:", z[0], "to", z[-1], "m")
print("Height sampling:", DZ, "m")

### Multilooking

In [ ]:
import numpy as np

# Target pixel spacing [m]
TARGET_RES = 200

# Reference acquisition metadata
annot = stack.annot_coreg[MASTER_IDX]

def get_xml_value(name):
    nodes = annot.xpath(
        f"//*[local-name()='sarImage']/*[local-name()='{name}']"
    )
    if not nodes:
        raise ValueError(f"Parameter '{name}' not found")
    return float(nodes[0].text)

# Original pixel spacing from BIOMASS metadata
az_spacing = get_xml_value("azimuthPixelSpacing")
rg_spacing = get_xml_value("rangePixelSpacing")

# Number of looks required
nlooks_az = int(np.ceil(TARGET_RES / az_spacing))
nlooks_rg = int(np.ceil(TARGET_RES / rg_spacing))

# Resulting pixel spacing
az_spacing_ml = nlooks_az * az_spacing
rg_spacing_ml = nlooks_rg * rg_spacing

# Resulting image dimensions
naz_ml = stack.nli // nlooks_az
nrg_ml = stack.nsa // nlooks_rg

print("ORIGINAL PIXEL SPACING")
print(f"Azimuth: {az_spacing:.2f} m")
print(f"Range:   {rg_spacing:.2f} m")

print("\nMULTILOOKING FACTORS")
print(f"Azimuth looks: {nlooks_az}")
print(f"Range looks:   {nlooks_rg}")
print(f"Total looks:   {nlooks_az * nlooks_rg}")

print("\nRESULTING PIXEL SPACING")
print(f"Azimuth: {az_spacing_ml:.2f} m")
print(f"Range:   {rg_spacing_ml:.2f} m")

print("\nRESULTING IMAGE DIMENSIONS")
print(f"Original:   {stack.nli} × {stack.nsa}")
print(f"Multilooked: {naz_ml} × {nrg_ml}")


### Apply Fourrier Beamformer algorithm

In [ ]:
import numpy as np
from tqdm.auto import tqdm

# ---------------------------------------------------------
# 1. Parameters
# ---------------------------------------------------------
looksa = nlooks_az
looksr = nlooks_rg

n_acq, n_az, n_rg = xp_stack.shape

# Height axis [m]
zaxis = np.linspace(-20, 100, 101)

# ---------------------------------------------------------
# 2. Define multilooked grid
# ---------------------------------------------------------
# Non-overlapping windows of looksa x looksr pixels
naz_ml = n_az // looksa
nrg_ml = n_rg // looksr

# Window starting positions
az_starts = np.arange(naz_ml) * looksa
rg_starts = np.arange(nrg_ml) * looksr

# Center pixel of each window
azax = az_starts + looksa // 2
rgax = rg_starts + looksr // 2

print(f"Original image: {n_az} x {n_rg}")
print(f"Multilooked grid: {naz_ml} x {nrg_ml}")
print(f"Number of looks: {looksa} x {looksr}")

# ---------------------------------------------------------
# 3. Get kz at the multilooked pixel centers
# ---------------------------------------------------------
# Shape: (acquisition, azimuth, range)
kz_ml = np.empty(
    (n_acq, naz_ml, nrg_ml),
    dtype=np.float32
)

for i, lut in enumerate(tqdm(stack.luts, desc="Interpolating kz")):

    kz_full = lut.get(
        "coregistration", "waveNumbers"
    ).upsampled

    # Extract kz at the multilooked pixel centers
    kz_ml[i] = np.ma.filled(
        np.ma.asarray(kz_full)[np.ix_(azax, rgax)],
        np.nan
    )

    del kz_full

# Mask invalid kz
kz_ml[
    ~np.isfinite(kz_ml) | (kz_ml == -9999)
] = np.nan

# ---------------------------------------------------------
# 4. Initialize Fourier tomographic cube
# ---------------------------------------------------------
cubeF = np.full(
    (naz_ml, nrg_ml, zaxis.size),
    np.nan,
    dtype=np.float32
)

# ---------------------------------------------------------
# 5. Fourier TomoSAR reconstruction
# ---------------------------------------------------------
for ia in tqdm(range(naz_ml), desc="Fourier TomoSAR"):

    az0 = az_starts[ia]
    az1 = az0 + looksa

    for ir in range(nrg_ml):

        rg0 = rg_starts[ir]
        rg1 = rg0 + looksr

        # ---------------------------------------------
        # Extract multilooking window
        # ---------------------------------------------
        # Shape: (acquisitions, azimuth looks, range looks)
        window = xp_stack[:, az0:az1, rg0:rg1]

        # Shape: (acquisitions, spatial looks)
        y = window.reshape(n_acq, -1)

        # Exclude invalid pixels
        valid = (
            np.all(np.isfinite(y), axis=0)
            & np.all(y != 0, axis=0)
        )

        y = y[:, valid]

        if y.shape[1] == 0:
            continue

        # ---------------------------------------------
        # Covariance matrix
        # ---------------------------------------------
        # Shape: (acquisitions, acquisitions)
        R = (y @ y.conj().T) / y.shape[1]

        # ---------------------------------------------
        # Vertical wavenumbers
        # ---------------------------------------------
        kkz = kz_ml[:, ia, ir]

        if not np.all(np.isfinite(kkz)):
            continue

        # ---------------------------------------------
        # Fourier steering matrix
        # ---------------------------------------------
        # Shape: (acquisitions, heights)
        amat = np.exp(
            -1j * np.outer(kkz, zaxis)
        )

        # ---------------------------------------------
        # Fourier reconstruction
        # P(z) = diag(A^H R A)
        # ---------------------------------------------
        spectrum = np.diag(
            np.linalg.multi_dot([
                amat.conj().T,
                R,
                amat
            ])
        ).real

        cubeF[ia, ir, :] = np.maximum(spectrum, 0)

# ---------------------------------------------------------
# 6. Results
# ---------------------------------------------------------
print("\nFourier TomoSAR reconstruction complete")
print("Cube shape:", cubeF.shape)
print(f"Cube memory: {cubeF.nbytes / 2**20:.1f} MiB")
print(f"Height range: {zaxis[0]} to {zaxis[-1]} m")

### Save Fourrier Cube

In [ ]:
import numpy as np
import json
from datetime import datetime, timezone

from biomass_tomosar.paths import processed_data_dir

# ---------------------------------------------------------
# 1. Output paths
# ---------------------------------------------------------
output_dir = processed_data_dir()
output_dir.mkdir(parents=True, exist_ok=True)

cube_file = output_dir / "fourier_cube.npy"
axes_file = output_dir / "fourier_axes.npz"
metadata_file = output_dir / "fourier_metadata.json"

# ---------------------------------------------------------
# 2. Save Fourier reconstruction
# ---------------------------------------------------------
np.save(cube_file, cubeF, allow_pickle=False)

# Save corresponding coordinate axes
np.savez(
    axes_file,
    azimuth_pixel_centers=azax,
    range_pixel_centers=rgax,
    height_m=zaxis,
)

# ---------------------------------------------------------
# 3. Retrieve pixel spacing from BIOMASS metadata
# ---------------------------------------------------------
annot = stack.annot_coreg[MASTER_IDX]

def xml_value(name):
    nodes = annot.xpath(
        f"//*[local-name()='sarImage']/*[local-name()='{name}']"
    )
    return float(nodes[0].text) if nodes and nodes[0].text else None

az_spacing = xml_value("azimuthPixelSpacing")
rg_spacing = xml_value("rangePixelSpacing")

# ---------------------------------------------------------
# 4. Save reconstruction metadata
# ---------------------------------------------------------
metadata = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "method": "Fourier TomoSAR",
    "polarization": "XP (HV + VH) / sqrt(2)",
    "source_slc": "xp_stack.npy",
    "phase_corrections": ["flattening", "SKP"],

    "acquisitions": {
        "count": int(n_acq),
        "reference_index": int(MASTER_IDX),
    },

    "reconstruction": {
        "cube_shape": list(cubeF.shape),
        "cube_dtype": str(cubeF.dtype),
        "axis_order": ["azimuth", "range", "height"],
        "height_min_m": float(zaxis[0]),
        "height_max_m": float(zaxis[-1]),
        "height_step_m": float(zaxis[1] - zaxis[0]),
        "height_samples": int(len(zaxis)),
        "output_quantity": "Fourier beamforming power",
        "method_description": "diag(A^H R A)",
        "invalid_value": "NaN",
    },

    "multilooking": {
        "azimuth_looks": int(looksa),
        "range_looks": int(looksr),
        "total_looks": int(looksa * looksr),
        "azimuth_pixel_spacing_m": az_spacing,
        "range_pixel_spacing_m": rg_spacing,
        "output_azimuth_spacing_m": (
            looksa * az_spacing if az_spacing is not None else None
        ),
        "output_range_spacing_m": (
            looksr * rg_spacing if rg_spacing is not None else None
        ),
    },

    "grid": {
        "original_shape": [int(n_az), int(n_rg)],
        "output_shape": [int(naz_ml), int(nrg_ml)],
        "axes_file": axes_file.name,
        "azimuth_coordinates": "Original SLC pixel-center indices",
        "range_coordinates": "Original SLC pixel-center indices",
        "height_coordinates": "Metres",
    },
}

with metadata_file.open("w") as f:
    json.dump(metadata, f, indent=4)

# ---------------------------------------------------------
# 5. Verify
# ---------------------------------------------------------
print("Fourier TomoSAR results saved")
print(f"Cube:     {cube_file}")
print(f"Axes:     {axes_file}")
print(f"Metadata: {metadata_file}")

print(f"\nCube shape: {cubeF.shape}")
print(f"Cube size: {cube_file.stat().st_size / 2**20:.1f} MiB")
print(f"Valid output pixels: {np.isfinite(cubeF).any(axis=2).sum():,}")

### Apply Capon algorithm

In [ ]:
import numpy as np
from tqdm.auto import tqdm

# ---------------------------------------------------------
# 1. Initialize Capon cube
# ---------------------------------------------------------
n_acq = xp_stack.shape[0]
naz_ml = len(az_starts)
nrg_ml = len(rg_starts)

DIAGONAL_LOADING = 0.05

cubeC = np.full(
    (naz_ml, nrg_ml, len(zaxis)),
    np.nan,
    dtype=np.float32
)

# ---------------------------------------------------------
# 2. Capon reconstruction
# ---------------------------------------------------------
for ia in tqdm(range(naz_ml), desc="Capon TomoSAR"):

    az0 = az_starts[ia]
    az1 = az0 + looksa

    for ir in range(nrg_ml):

        rg0 = rg_starts[ir]
        rg1 = rg0 + looksr

        # ---------------------------------------------
        # Extract multilooking window
        # ---------------------------------------------
        window = xp_stack[:, az0:az1, rg0:rg1]

        # Shape: (acquisitions, spatial looks)
        y = window.reshape(n_acq, -1)

        valid = (
            np.all(np.isfinite(y), axis=0)
            & np.all(y != 0, axis=0)
        )

        y = y[:, valid]

        if y.shape[1] == 0:
            continue

        # ---------------------------------------------
        # Covariance matrix
        # ---------------------------------------------
        R = (y @ y.conj().T) / y.shape[1]

        # ---------------------------------------------
        # Vertical steering matrix
        # ---------------------------------------------
        kkz = kz_ml[:, ia, ir]

        if not np.all(np.isfinite(kkz)):
            continue

        amat = np.exp(
            -1j * np.outer(kkz, zaxis)
        )

        # ---------------------------------------------
        # Diagonal loading
        # ---------------------------------------------
        mean_power = np.trace(R).real / n_acq

        if not np.isfinite(mean_power) or mean_power <= 0:
            continue

        R_loaded = R + (
            DIAGONAL_LOADING
            * mean_power
            * np.eye(n_acq)
        )

        # ---------------------------------------------
        # Capon weights
        # ---------------------------------------------
        # Equivalent to R_loaded^-1 @ amat
        wnum = np.linalg.solve(R_loaded, amat)

        # a^H R_loaded^-1 a for each height
        wden = np.sum(
            amat.conj() * wnum,
            axis=0
        ).real

        valid_heights = np.isfinite(wden) & (wden > 0)

        if not np.any(valid_heights):
            continue

        # ---------------------------------------------
        # Capon power spectrum
        # ---------------------------------------------
        # For distortionless minimum-variance weights:
        # w^H R_loaded w = 1 / (a^H R_loaded^-1 a)
        cubeC[ia, ir, valid_heights] = (
            1.0 / wden[valid_heights]
        ).astype(np.float32)

# ---------------------------------------------------------
# 3. Results
# ---------------------------------------------------------
print("\nCapon TomoSAR reconstruction complete")
print("Cube shape:", cubeC.shape)
print(f"Cube memory: {cubeC.nbytes / 2**20:.1f} MiB")
print(f"Diagonal loading: {DIAGONAL_LOADING * 100:.1f}%")

### Save Data Cubes

In [ ]:
import numpy as np
import json
from datetime import datetime, timezone

from biomass_tomosar.paths import processed_data_dir

# ---------------------------------------------------------
# 1. Output directory and unique timestamp
# ---------------------------------------------------------
output_dir = processed_data_dir()
output_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f_UTC")

fourier_file = output_dir / f"fourier_cube_{timestamp}.npy"
capon_file = output_dir / f"capon_cube_{timestamp}.npy"
axes_file = output_dir / f"tomosar_axes_{timestamp}.npz"
metadata_file = output_dir / f"tomosar_metadata_{timestamp}.json"

# ---------------------------------------------------------
# 2. Validate reconstruction dimensions
# ---------------------------------------------------------
assert cubeF.shape == cubeC.shape
assert cubeF.shape == (len(azax), len(rgax), len(zaxis))

# ---------------------------------------------------------
# 3. Save reconstruction cubes
# ---------------------------------------------------------
# 'xb' ensures an existing file can never be overwritten
with fourier_file.open("xb") as f:
    np.save(f, cubeF, allow_pickle=False)

with capon_file.open("xb") as f:
    np.save(f, cubeC, allow_pickle=False)

# ---------------------------------------------------------
# 4. Save reconstruction axes
# ---------------------------------------------------------
with axes_file.open("xb") as f:
    np.savez(
        f,
        azimuth_pixel_centers=azax,
        range_pixel_centers=rgax,
        height_m=zaxis,
    )

# ---------------------------------------------------------
# 5. Extract pixel spacing from BIOMASS metadata
# ---------------------------------------------------------
annot = stack.annot_coreg[MASTER_IDX]

def xml_value(name):
    nodes = annot.xpath(
        f"//*[local-name()='sarImage']/*[local-name()='{name}']"
    )
    return float(nodes[0].text) if nodes and nodes[0].text else None

az_spacing = xml_value("azimuthPixelSpacing")
rg_spacing = xml_value("rangePixelSpacing")

# ---------------------------------------------------------
# 6. Build metadata
# ---------------------------------------------------------
metadata = {
    "timestamp_utc": timestamp,

    "files": {
        "fourier": fourier_file.name,
        "capon": capon_file.name,
        "axes": axes_file.name,
    },

    "source": {
        "slc_file": "xp_stack.npy",
        "polarization": "(HV + VH) / sqrt(2)",
        "phase_corrections": ["flattening", "SKP"],
        "n_acquisitions": int(xp_stack.shape[0]),
        "reference_acquisition_index": int(MASTER_IDX),
    },

    "reconstruction": {
        "methods": ["Fourier", "Capon"],
        "cube_shape": list(cubeF.shape),
        "axis_order": ["azimuth", "range", "height"],
        "fourier_quantity": "Beamforming power",
        "capon_quantity": "Loaded MVDR power",
        "capon_diagonal_loading": float(DIAGONAL_LOADING),
        "height_min_m": float(zaxis[0]),
        "height_max_m": float(zaxis[-1]),
        "height_step_m": float(zaxis[1] - zaxis[0]),
        "height_samples": int(len(zaxis)),
        "invalid_value": "NaN",
    },

    "multilooking": {
        "azimuth_looks": int(looksa),
        "range_looks": int(looksr),
        "total_looks": int(looksa * looksr),
        "original_azimuth_spacing_m": az_spacing,
        "original_range_spacing_m": rg_spacing,
        "output_azimuth_spacing_m": looksa * az_spacing if az_spacing is not None else None,
        "output_range_spacing_m": looksr * rg_spacing if rg_spacing is not None else None,
    },

    "grid": {
        "azimuth_pixels": int(len(azax)),
        "range_pixels": int(len(rgax)),
        "height_pixels": int(len(zaxis)),
        "axes_file": axes_file.name,
    },
}

with metadata_file.open("x") as f:
    json.dump(metadata, f, indent=4)

# ---------------------------------------------------------
# 7. Verify
# ---------------------------------------------------------
print("TomoSAR reconstructions saved successfully")
print(f"\nTimestamp: {timestamp}")

for file in [fourier_file, capon_file, axes_file, metadata_file]:
    print(f"{file.name}: {file.stat().st_size / 2**20:.2f} MiB")

print(f"\nCube shape: {cubeF.shape}")
print(f"Saved to: {output_dir}")

### Load Data Cubes

In [ ]:
import numpy as np
import json
from biomass_tomosar.paths import processed_data_dir

timestamp = "20261122_213343_042797_UTC"  # Replace with your saved timestamp

data_dir = processed_data_dir()

cubeF = np.load(
    data_dir / f"fourier_cube_{timestamp}.npy",
    mmap_mode="r"
)

cubeC = np.load(
    data_dir / f"capon_cube_{timestamp}.npy",
    mmap_mode="r"
)

with np.load(data_dir / f"tomosar_axes_{timestamp}.npz") as axes:
    azax = axes["azimuth_pixel_centers"]
    rgax = axes["range_pixel_centers"]
    zaxis = axes["height_m"]

with (data_dir / f"tomosar_metadata_{timestamp}.json").open() as f:
    metadata = json.load(f)

print("Fourier:", cubeF.shape)
print("Capon:", cubeC.shape)
print("Height axis:", zaxis[0], "to", zaxis[-1], "m")

### Range Cut Against Lidar

In [ ]:
import importlib
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import biomass_tomosar.paths as paths

# Reload paths.py so the new outputs_dir() is available
# without restarting the Jupyter kernel.
importlib.reload(paths)

# =========================================================
# 1. USER SETTINGS
# =========================================================
ACQ_IDX = 0
CUT_RANGE_KM = 768          # None = center of SAR/GEDI overlap
SLC_DISPLAY_STEP = 6

NORMALIZE_PROFILES = False  # OFF by default
DYNAMIC_RANGE_DB = 30

SAVE_FIGURES = True
FIGURE_DPI = 200

# =========================================================
# 2. LOAD GEDI DATA
# =========================================================
gedi_dir = paths.gedi_data_dir()

gedi_height = np.load(gedi_dir / "gedi_forestHeight.npy")

grid = pd.read_csv(
    gedi_dir / "gedi_forestHeight.grd"
).iloc[0]

az0 = float(grid["az0"])
naz = int(grid["naz"])
daz = float(grid["daz"])

rg0 = float(grid["rg0"])
nrg = int(grid["nrg"])
drg = float(grid["drg"])

gedi_az = az0 + daz * np.arange(naz)
gedi_rg = rg0 + drg * np.arange(nrg)

assert gedi_height.shape == (naz, nrg)

gedi_points = pd.read_csv(
    gedi_dir / "gedi.csv",
    usecols=["rg", "az", "rh_98", "rh_0"],
)

gedi_points["forest_height"] = (
    gedi_points["rh_98"] - gedi_points["rh_0"]
)

# =========================================================
# 3. SAR COORDINATES FROM BIOMASS METADATA
# =========================================================
annot = stack.annot_coreg[MASTER_IDX]

def xml_float(name):
    nodes = annot.xpath(
        f"//*[local-name()='sarImage']/*[local-name()='{name}']"
    )
    if not nodes:
        raise ValueError(f"Missing SAR metadata field: {name}")
    return float(nodes[0].text)

C = 299_792_458.0

az_spacing = xml_float("azimuthPixelSpacing")
rg_spacing = xml_float("rangePixelSpacing")
first_range_time = xml_float("firstSampleSlantRangeTime")

first_range_m = C * first_range_time / 2

n_acq, n_az, n_rg = xp_stack.shape

sar_az = np.arange(n_az) * az_spacing
sar_rg = first_range_m + np.arange(n_rg) * rg_spacing

# Coordinates of the multilooked TomoSAR pixels
tomo_az = sar_az[np.asarray(azax, dtype=int)]
tomo_rg = sar_rg[np.asarray(rgax, dtype=int)]

assert cubeF.shape == cubeC.shape
assert cubeF.shape == (len(tomo_az), len(tomo_rg), len(zaxis))

# =========================================================
# 4. SELECT RANGE CUT
# =========================================================
range_start = max(tomo_rg[0], gedi_rg[0])
range_end = min(tomo_rg[-1], gedi_rg[-1])

if range_start >= range_end:
    raise ValueError("SAR and GEDI range grids do not overlap.")

if CUT_RANGE_KM is None:
    target_range_m = (range_start + range_end) / 2
else:
    target_range_m = CUT_RANGE_KM * 1000

if not (range_start <= target_range_m <= range_end):
    raise ValueError(
        f"Selected cut is outside the overlap: "
        f"{range_start/1000:.3f}–{range_end/1000:.3f} km"
    )

RG_IDX = int(np.argmin(np.abs(tomo_rg - target_range_m)))
cut_range_m = tomo_rg[RG_IDX]

GEDI_RG_IDX = int(np.argmin(np.abs(gedi_rg - cut_range_m)))

print(f"Selected range: {cut_range_m / 1000:.3f} km")
print(f"TomoSAR range index: {RG_IDX}")
print(f"Normalization: {'ON' if NORMALIZE_PROFILES else 'OFF'}")

# =========================================================
# 5. FIGURE 1 — SLC OVERVIEW
# =========================================================
slc = xp_stack[
    ACQ_IDX,
    ::SLC_DISPLAY_STEP,
    ::SLC_DISPLAY_STEP,
]

amplitude = np.abs(slc)
amplitude_db = 20 * np.log10(np.maximum(amplitude, 1e-8))

valid_amp = amplitude_db[
    np.isfinite(amplitude_db) & (amplitude > 0)
]

if valid_amp.size:
    vmin_slc, vmax_slc = np.percentile(valid_amp, [2, 98])
else:
    vmin_slc, vmax_slc = -40, 0

fig_slc, ax = plt.subplots(
    figsize=(9, 11),
    constrained_layout=True,
)

im_slc = ax.imshow(
    amplitude_db,
    extent=[
        sar_rg[0] / 1000,
        sar_rg[-1] / 1000,
        sar_az[0] / 1000,
        sar_az[-1] / 1000,
    ],
    origin="lower",
    aspect="auto",
    cmap="gray",
    vmin=vmin_slc,
    vmax=vmax_slc,
)

ax.axvline(
    cut_range_m / 1000,
    color="red",
    linewidth=2,
    label=f"Range cut: {cut_range_m / 1000:.3f} km",
)

ax.scatter(
    gedi_points["rg"].iloc[::10] / 1000,
    gedi_points["az"].iloc[::10] / 1000,
    s=1,
    color="cyan",
    alpha=0.25,
    rasterized=True,
    label="GEDI tracks",
)

ax.set_title(f"Corrected XP SLC — Acquisition {ACQ_IDX + 1}")
ax.set_xlabel("Slant range [km]")
ax.set_ylabel("Azimuth [km]")

ax.set_xlim(sar_rg[0] / 1000, sar_rg[-1] / 1000)
ax.set_ylim(sar_az[-1] / 1000, sar_az[0] / 1000)

ax.legend(loc="upper right")

fig_slc.colorbar(
    im_slc,
    ax=ax,
    label="SLC amplitude [dB]",
    shrink=0.7,
)

# =========================================================
# 6. EXTRACT TOMOGRAPHIC SECTIONS
# =========================================================
fourier_section = cubeF[:, RG_IDX, :]
capon_section = cubeC[:, RG_IDX, :]


def prepare_power_db(section, normalize=False):
    """
    Convert reconstructed power to dB.

    normalize=False:
        Original power in dB.

    normalize=True:
        Normalize each azimuth profile by its peak power.
    """
    power = np.asarray(section, dtype=np.float32)

    valid = np.isfinite(power) & (power > 0)

    if normalize:
        peaks = np.max(
            np.where(valid, power, 0),
            axis=1,
            keepdims=True,
        )

        relative = np.full_like(power, np.nan)

        np.divide(
            power,
            peaks,
            out=relative,
            where=valid & (peaks > 0),
        )

        power = relative
        valid = np.isfinite(power) & (power > 0)

    power_db = np.full_like(power, np.nan)

    power_db[valid] = 10 * np.log10(power[valid])

    # Return (height, azimuth)
    return power_db.T


fourier_db = prepare_power_db(
    fourier_section,
    normalize=NORMALIZE_PROFILES,
)

capon_db = prepare_power_db(
    capon_section,
    normalize=NORMALIZE_PROFILES,
)

# =========================================================
# 7. COLOR LIMITS
# =========================================================
def get_color_limits(power_db):
    if NORMALIZE_PROFILES:
        return -DYNAMIC_RANGE_DB, 0

    finite = power_db[np.isfinite(power_db)]

    if finite.size == 0:
        return -DYNAMIC_RANGE_DB, 0

    vmax = np.percentile(finite, 99)
    vmin = vmax - DYNAMIC_RANGE_DB

    return vmin, vmax


fourier_vmin, fourier_vmax = get_color_limits(fourier_db)
capon_vmin, capon_vmax = get_color_limits(capon_db)

# =========================================================
# 8. GEDI REFERENCE
# =========================================================
gedi_profile = gedi_height[:, GEDI_RG_IDX]

range_step_ml = (
    np.median(np.diff(tomo_rg))
    if len(tomo_rg) > 1
    else 60.0
)

gedi_half_width_m = max(30.0, range_step_ml / 2)

gedi_near_cut = gedi_points[
    np.abs(gedi_points["rg"] - cut_range_m)
    <= gedi_half_width_m
]

# =========================================================
# 9. FIGURE 2 — FOURIER / CAPON TOMOGRAMS
# =========================================================
fig_tomo, axes = plt.subplots(
    2, 1,
    figsize=(15, 10),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)

sections = [
    ("Fourier TomoSAR", fourier_db, fourier_vmin, fourier_vmax),
    ("Capon TomoSAR", capon_db, capon_vmin, capon_vmax),
]

for ax, (title, section_db, vmin, vmax) in zip(axes, sections):

    im = ax.pcolormesh(
        tomo_az / 1000,
        zaxis,
        section_db,
        shading="nearest",
        cmap="magma",
        vmin=vmin,
        vmax=vmax,
        rasterized=True,
    )

    # Interpolated GEDI canopy height
    ax.plot(
        gedi_az / 1000,
        gedi_profile,
        color="cyan",
        linewidth=1.8,
        label="GEDI canopy height",
        zorder=4,
    )

    # Raw GEDI measurements near the cut
    if len(gedi_near_cut):
        ax.scatter(
            gedi_near_cut["az"] / 1000,
            gedi_near_cut["forest_height"],
            s=8,
            color="lime",
            alpha=0.55,
            label="Raw GEDI",
            rasterized=True,
            zorder=5,
        )

    ax.set_title(title)
    ax.set_ylabel("Height [m]")
    ax.set_ylim(zaxis[0], zaxis[-1])
    ax.set_xlim(tomo_az[0] / 1000, tomo_az[-1] / 1000)

    ax.grid(alpha=0.15)
    ax.legend(loc="upper right")

    fig_tomo.colorbar(
        im,
        ax=ax,
        label=(
            "Relative power [dB]"
            if NORMALIZE_PROFILES
            else "Reconstructed power [dB]"
        ),
        shrink=0.85,
    )

axes[-1].set_xlabel("Azimuth [km]")

mode = "Normalized" if NORMALIZE_PROFILES else "Unnormalized"

fig_tomo.suptitle(
    f"Fourier / Capon — {mode} power\n"
    f"Range: {cut_range_m / 1000:.3f} km",
    fontsize=14,
)

# =========================================================
# 10. SAVE FIGURES TO A NEW TIMESTAMPED FOLDER
# =========================================================
if SAVE_FIGURES:

    # Root output directory from config/local.toml
    output_root = paths.outputs_dir()

    # Unique timestamp for this cut
    timestamp = datetime.now(timezone.utc).strftime(
        "%Y%m%d_%H%M%S_%f_UTC"
    )

    # One folder per visualization run
    run_dir = output_root / f"tomogram_{timestamp}"

    # exist_ok=False prevents reuse of an existing folder
    run_dir.mkdir(parents=True, exist_ok=False)

    slc_figure_file = run_dir / "slc_overview.png"
    tomo_figure_file = run_dir / "fourier_capon_gedi.png"

    # Exclusive file creation prevents overwriting
    with slc_figure_file.open("xb") as f:
        fig_slc.savefig(
            f,
            format="png",
            dpi=FIGURE_DPI,
            bbox_inches="tight",
        )

    with tomo_figure_file.open("xb") as f:
        fig_tomo.savefig(
            f,
            format="png",
            dpi=FIGURE_DPI,
            bbox_inches="tight",
        )

    print("\nFigures saved:")
    print(f"  {slc_figure_file}")
    print(f"  {tomo_figure_file}")

# =========================================================
# 11. DISPLAY
# =========================================================
plt.show()

### Azimuth Cut against Lidar

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timezone

from biomass_tomosar.paths import outputs_dir

# =========================================================
# 1. SETTINGS
# =========================================================
ACQ_IDX = 0

CUT_AZ_KM = 80           # None = center of SAR/GEDI overlap
SLC_DISPLAY_STEP = 6

NORMALIZE_PROFILES = False
DYNAMIC_RANGE_DB = 30

SAVE_FIGURES = True
FIGURE_DPI = 200

# =========================================================
# 2. SELECT AZIMUTH CUT
# =========================================================
az_start = max(tomo_az[0], gedi_az[0])
az_end = min(tomo_az[-1], gedi_az[-1])

if az_start >= az_end:
    raise ValueError("SAR and GEDI azimuth grids do not overlap.")

if CUT_AZ_KM is None:
    target_az_m = (az_start + az_end) / 2
else:
    target_az_m = CUT_AZ_KM * 1000

if not (az_start <= target_az_m <= az_end):
    raise ValueError(
        f"Selected azimuth is outside the overlap: "
        f"{az_start/1000:.3f}–{az_end/1000:.3f} km"
    )

# Nearest TomoSAR azimuth sample
AZ_IDX = int(np.argmin(np.abs(tomo_az - target_az_m)))
cut_az_m = tomo_az[AZ_IDX]

# Nearest GEDI azimuth sample
GEDI_AZ_IDX = int(np.argmin(np.abs(gedi_az - cut_az_m)))

print(f"Selected azimuth: {cut_az_m / 1000:.3f} km")
print(f"TomoSAR azimuth index: {AZ_IDX}")
print(f"GEDI azimuth: {gedi_az[GEDI_AZ_IDX] / 1000:.3f} km")
print(f"Normalization: {'ON' if NORMALIZE_PROFILES else 'OFF'}")

# =========================================================
# 3. FIGURE 1 — SLC OVERVIEW
# =========================================================
slc = xp_stack[
    ACQ_IDX,
    ::SLC_DISPLAY_STEP,
    ::SLC_DISPLAY_STEP,
]

amplitude = np.abs(slc)
amplitude_db = 20 * np.log10(np.maximum(amplitude, 1e-8))

valid_amp = amplitude_db[
    np.isfinite(amplitude_db) & (amplitude > 0)
]

if valid_amp.size:
    vmin_slc, vmax_slc = np.percentile(valid_amp, [2, 98])
else:
    vmin_slc, vmax_slc = -40, 0

fig_slc, ax = plt.subplots(
    figsize=(9, 11),
    constrained_layout=True,
)

im_slc = ax.imshow(
    amplitude_db,
    extent=[
        sar_rg[0] / 1000,
        sar_rg[-1] / 1000,
        sar_az[0] / 1000,
        sar_az[-1] / 1000,
    ],
    origin="lower",
    aspect="auto",
    cmap="gray",
    vmin=vmin_slc,
    vmax=vmax_slc,
)

# Horizontal line for the selected azimuth
ax.axhline(
    cut_az_m / 1000,
    color="red",
    linewidth=2,
    label=f"Azimuth cut: {cut_az_m / 1000:.3f} km",
)

# GEDI tracks
ax.scatter(
    gedi_points["rg"].iloc[::10] / 1000,
    gedi_points["az"].iloc[::10] / 1000,
    s=1,
    color="cyan",
    alpha=0.25,
    rasterized=True,
    label="GEDI tracks",
)

ax.set_title(f"Corrected XP SLC — Acquisition {ACQ_IDX + 1}")
ax.set_xlabel("Slant range [km]")
ax.set_ylabel("Azimuth [km]")

ax.set_xlim(sar_rg[0] / 1000, sar_rg[-1] / 1000)
ax.set_ylim(sar_az[-1] / 1000, sar_az[0] / 1000)

ax.legend(loc="upper right")

fig_slc.colorbar(
    im_slc,
    ax=ax,
    label="SLC amplitude [dB]",
    shrink=0.7,
)

# =========================================================
# 4. EXTRACT HEIGHT VS RANGE SECTIONS
# =========================================================
# cube shape: (azimuth, range, height)
#
# Fixed azimuth -> section shape: (range, height)

fourier_section = cubeF[AZ_IDX, :, :]
capon_section = cubeC[AZ_IDX, :, :]


def prepare_power_db(section, normalize=False):
    """
    Convert reconstructed power to dB.

    normalize=False:
        Preserve original reconstructed power.

    normalize=True:
        Normalize each range profile by its maximum
        along the height dimension.

    Returns an array of shape (height, range).
    """
    power = np.asarray(section, dtype=np.float32)
    valid = np.isfinite(power) & (power > 0)

    if normalize:
        peaks = np.max(
            np.where(valid, power, 0),
            axis=1,
            keepdims=True,
        )

        relative = np.full_like(power, np.nan)

        np.divide(
            power,
            peaks,
            out=relative,
            where=valid & (peaks > 0),
        )

        power = relative
        valid = np.isfinite(power) & (power > 0)

    power_db = np.full_like(power, np.nan)
    power_db[valid] = 10 * np.log10(power[valid])

    return power_db.T


fourier_db = prepare_power_db(
    fourier_section,
    normalize=NORMALIZE_PROFILES,
)

capon_db = prepare_power_db(
    capon_section,
    normalize=NORMALIZE_PROFILES,
)

# =========================================================
# 5. COLOR LIMITS
# =========================================================
def get_color_limits(power_db):
    if NORMALIZE_PROFILES:
        return -DYNAMIC_RANGE_DB, 0

    finite = power_db[np.isfinite(power_db)]

    if finite.size == 0:
        return -DYNAMIC_RANGE_DB, 0

    vmax = np.percentile(finite, 99)
    vmin = vmax - DYNAMIC_RANGE_DB

    return vmin, vmax


fourier_vmin, fourier_vmax = get_color_limits(fourier_db)
capon_vmin, capon_vmax = get_color_limits(capon_db)

# =========================================================
# 6. GEDI PROFILE AT SELECTED AZIMUTH
# =========================================================
# Interpolated forest height along range
gedi_profile = gedi_height[GEDI_AZ_IDX, :]

# Raw GEDI points in a strip around the selected azimuth
az_step_ml = (
    np.median(np.diff(tomo_az))
    if len(tomo_az) > 1
    else 60.0
)

gedi_half_width_m = max(30.0, az_step_ml / 2)

gedi_near_cut = gedi_points[
    np.abs(gedi_points["az"] - cut_az_m)
    <= gedi_half_width_m
]

# =========================================================
# 7. FIGURE 2 — FOURIER / CAPON HEIGHT VS RANGE
# =========================================================
fig_tomo, axes = plt.subplots(
    2, 1,
    figsize=(15, 10),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)

sections = [
    ("Fourier TomoSAR", fourier_db, fourier_vmin, fourier_vmax),
    ("Capon TomoSAR", capon_db, capon_vmin, capon_vmax),
]

for ax, (title, section_db, vmin, vmax) in zip(axes, sections):

    im = ax.pcolormesh(
        tomo_rg / 1000,
        zaxis,
        section_db,
        shading="nearest",
        cmap="magma",
        vmin=vmin,
        vmax=vmax,
        rasterized=True,
    )

    # Interpolated GEDI canopy-height profile
    ax.plot(
        gedi_rg / 1000,
        gedi_profile,
        color="cyan",
        linewidth=1.8,
        label="GEDI canopy height",
        zorder=4,
    )

    # Raw GEDI measurements near the cut
    if len(gedi_near_cut):
        ax.scatter(
            gedi_near_cut["rg"] / 1000,
            gedi_near_cut["forest_height"],
            s=8,
            color="lime",
            alpha=0.55,
            rasterized=True,
            label="Raw GEDI",
            zorder=5,
        )

    ax.set_title(title)
    ax.set_ylabel("Height [m]")
    ax.set_ylim(zaxis[0], zaxis[-1])
    ax.set_xlim(tomo_rg[0] / 1000, tomo_rg[-1] / 1000)

    ax.grid(alpha=0.15)
    ax.legend(loc="upper right")

    fig_tomo.colorbar(
        im,
        ax=ax,
        label=(
            "Relative power [dB]"
            if NORMALIZE_PROFILES
            else "Reconstructed power [dB]"
        ),
        shrink=0.85,
    )

axes[-1].set_xlabel("Slant range [km]")

mode = "Normalized" if NORMALIZE_PROFILES else "Unnormalized"

fig_tomo.suptitle(
    f"Fourier / Capon — {mode} power\n"
    f"Azimuth: {cut_az_m / 1000:.3f} km",
    fontsize=14,
)

# =========================================================
# 8. SAVE FIGURES IN A NEW TIMESTAMPED FOLDER
# =========================================================
if SAVE_FIGURES:

    output_root = outputs_dir()

    timestamp = datetime.now(timezone.utc).strftime(
        "%Y%m%d_%H%M%S_%f_UTC"
    )

    run_dir = output_root / f"tomogram_azimuth_{timestamp}"
    run_dir.mkdir(parents=True, exist_ok=False)

    slc_file = run_dir / "slc_overview.png"
    tomo_file = run_dir / "fourier_capon_gedi.png"

    with slc_file.open("xb") as f:
        fig_slc.savefig(
            f,
            format="png",
            dpi=FIGURE_DPI,
            bbox_inches="tight",
        )

    with tomo_file.open("xb") as f:
        fig_tomo.savefig(
            f,
            format="png",
            dpi=FIGURE_DPI,
            bbox_inches="tight",
        )

    print("\nFigures saved:")
    print(f"  {slc_file}")
    print(f"  {tomo_file}")

# =========================================================
# 9. DISPLAY
# =========================================================
plt.show()

###

### Pixel Cut Against Lidar

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timezone

from biomass_tomosar.paths import outputs_dir

# =========================================================
# 1. SETTINGS
# =========================================================
AZ_KM = 40                 # None = center of scene
RG_KM = 765                 # None = center of scene

NORMALIZE_PROFILES = False   # False = original power in dB
SAVE_FIGURE = True
FIGURE_DPI = 200

# =========================================================
# 2. SELECT NEAREST TOMOSAR PIXEL
# =========================================================
target_az = (
    np.mean([tomo_az[0], tomo_az[-1]])
    if AZ_KM is None else AZ_KM * 1000
)

target_rg = (
    np.mean([tomo_rg[0], tomo_rg[-1]])
    if RG_KM is None else RG_KM * 1000
)

AZ_IDX = int(np.argmin(np.abs(tomo_az - target_az)))
RG_IDX = int(np.argmin(np.abs(tomo_rg - target_rg)))

selected_az = tomo_az[AZ_IDX]
selected_rg = tomo_rg[RG_IDX]

print(f"Azimuth: {selected_az / 1000:.3f} km (index {AZ_IDX})")
print(f"Range:   {selected_rg / 1000:.3f} km (index {RG_IDX})")

# =========================================================
# 3. EXTRACT FOURIER AND CAPON PROFILES
# =========================================================
fourier_power = np.asarray(cubeF[AZ_IDX, RG_IDX, :])
capon_power = np.asarray(cubeC[AZ_IDX, RG_IDX, :])


def power_to_db(power, normalize=False):
    power = np.asarray(power, dtype=np.float64)

    # Ignore invalid and non-positive power values
    valid = np.isfinite(power) & (power > 0)
    result = np.full(power.shape, np.nan)

    if not np.any(valid):
        return result

    if normalize:
        power = power / np.max(power[valid])

    result[valid] = 10 * np.log10(power[valid])
    return result


fourier_db = power_to_db(fourier_power, NORMALIZE_PROFILES)
capon_db = power_to_db(capon_power, NORMALIZE_PROFILES)

# =========================================================
# 4. OPTIONAL GEDI REFERENCE
# =========================================================
gedi_height_at_pixel = None

if all(name in globals() for name in ("gedi_height", "gedi_az", "gedi_rg")):
    gedi_az_idx = int(np.argmin(np.abs(gedi_az - selected_az)))
    gedi_rg_idx = int(np.argmin(np.abs(gedi_rg - selected_rg)))

    height = gedi_height[gedi_az_idx, gedi_rg_idx]

    if np.isfinite(height):
        gedi_height_at_pixel = float(height)

# =========================================================
# 6. FIGURE 1 — SLC OVERVIEW
# =========================================================
fig_slc, ax = plt.subplots(figsize=(9, 11), constrained_layout=True)

im = ax.imshow(
    amplitude_db,
    extent=[
        sar_rg[0] / 1000,
        sar_rg[-1] / 1000,
        sar_az[0] / 1000,
        sar_az[-1] / 1000,
    ],
    origin="lower",
    cmap="gray",
    aspect="auto",
    vmin=vmin_slc,
    vmax=vmax_slc,
)

ax.plot(
    selected_rg / 1000,
    selected_az / 1000,
    marker="x",
    color="red",
    markersize=14,
    markeredgewidth=3,
    linestyle="none",
    label="Selected pixel",
    zorder=5,
)

ax.set_title(f"XP SLC — Acquisition {ACQ_IDX + 1}")
ax.set_xlabel("Slant range [km]")
ax.set_ylabel("Azimuth [km]")
ax.set_ylim(sar_az[-1] / 1000, sar_az[0] / 1000)
ax.legend(loc="upper right")

fig_slc.colorbar(
    im, ax=ax, label="SLC amplitude [dB]", shrink=0.8
)

# =========================================================
# 7. FIGURE 2 — VERTICAL POWER PROFILES
# =========================================================
fig_profile, ax = plt.subplots(
    figsize=(10, 6), constrained_layout=True
)

ax.plot(zaxis, fourier_db, linewidth=2, label="Fourier")
ax.plot(zaxis, capon_db, linewidth=2, label="Capon")

if gedi_height_at_pixel is not None:
    ax.axvline(
        gedi_height_at_pixel,
        color="green",
        linestyle="--",
        linewidth=1.8,
        label=f"GEDI canopy height: {gedi_height_at_pixel:.1f} m",
    )

ax.set_title(
    f"Vertical TomoSAR Power Profile\n"
    f"Azimuth = {selected_az / 1000:.3f} km, "
    f"Range = {selected_rg / 1000:.3f} km"
)
ax.set_xlabel("Height [m]")
ax.set_ylabel(
    "Relative power [dB]"
    if NORMALIZE_PROFILES
    else "Reconstructed power [dB]"
)
ax.set_xlim(zaxis[0], zaxis[-1])
ax.grid(True, alpha=0.3)
ax.legend()

# =========================================================
# 8. SAVE BOTH FIGURES
# =========================================================
if SAVE_FIGURE:
    timestamp = datetime.now(timezone.utc).strftime(
        "%Y%m%d_%H%M%S_%f_UTC"
    )

    output_dir = outputs_dir() / f"pixel_profile_{timestamp}"
    output_dir.mkdir(parents=True, exist_ok=False)

    slc_file = output_dir / "slc_overview.png"
    profile_file = output_dir / "fourier_capon_profile.png"

    with slc_file.open("xb") as f:
        fig_slc.savefig(
            f, format="png", dpi=FIGURE_DPI, bbox_inches="tight"
        )

    with profile_file.open("xb") as f:
        fig_profile.savefig(
            f, format="png", dpi=FIGURE_DPI, bbox_inches="tight"
        )

    print(f"Figures saved in: {output_dir}")

plt.show()

## 2D Height heat Map & 3D Height map

### Reload Data

In [ ]:
import json
import numpy as np
import pandas as pd

from biomass_tomosar.paths import (
    biomass_stack_dir,
    processed_data_dir,
    gedi_data_dir,
)
from biomass_tomosar.bm_data_loader import bmStack

# ---------------------------------------------------------
# 1. Load saved reconstruction axes
# ---------------------------------------------------------
processed_dir = processed_data_dir()

# Use the reconstruction metadata already loaded in the notebook
# as 'metadata'. Otherwise, read the most recent metadata file.
if (
    "metadata" not in globals()
    or "files" not in metadata
):
    metadata_files = sorted(
        processed_dir.glob("tomosar_metadata_*.json")
    )

    if not metadata_files:
        raise FileNotFoundError("No TomoSAR metadata found.")

    with metadata_files[-1].open() as f:
        metadata = json.load(f)

axes_file = processed_dir / metadata["files"]["axes"]

with np.load(axes_file) as axes:
    azax = axes["azimuth_pixel_centers"].astype(int)
    rgax = axes["range_pixel_centers"].astype(int)
    zaxis = axes["height_m"]

# ---------------------------------------------------------
# 2. Initialize bmStack for annotations only
# ---------------------------------------------------------
with (processed_dir / "xp_stack_metadata.json").open() as f:
    xp_metadata = json.load(f)

products = [
    biomass_stack_dir() / name
    for name in xp_metadata["acquisitions"]
]

MASTER_IDX = xp_metadata["reference_acquisition_index"]

# Only reads metadata and LUTs, NOT full-resolution SLCs
stack = bmStack([str(p) for p in products])

# ---------------------------------------------------------
# 3. Retrieve SAR image grid parameters
# ---------------------------------------------------------
annot = stack.annot_coreg[MASTER_IDX]

def xml_float(name):
    nodes = annot.xpath(
        f"//*[local-name()='sarImage']/*[local-name()='{name}']"
    )
    if not nodes or nodes[0].text is None:
        raise ValueError(f"Missing SAR metadata: {name}")
    return float(nodes[0].text)

C = 299_792_458.0

az_spacing = xml_float("azimuthPixelSpacing")
rg_spacing = xml_float("rangePixelSpacing")
first_range_time = xml_float("firstSampleSlantRangeTime")

first_range_m = C * first_range_time / 2

# ---------------------------------------------------------
# 4. Reconstruct physical coordinate axes [m]
# ---------------------------------------------------------
# Original full-resolution SAR grid
sar_az = np.arange(stack.nli) * az_spacing
sar_rg = first_range_m + np.arange(stack.nsa) * rg_spacing

# Multilooked TomoSAR grid at the saved pixel centers
tomo_az = sar_az[azax]
tomo_rg = sar_rg[rgax]

# ---------------------------------------------------------
# 5. Load GEDI height map and grid
# ---------------------------------------------------------
gedi_dir = gedi_data_dir()

gedi_height = np.load(
    gedi_dir / "gedi_forestHeight.npy",
    mmap_mode="r"
)

grid = pd.read_csv(
    gedi_dir / "gedi_forestHeight.grd"
).iloc[0]

az0 = float(grid["az0"])
naz = int(grid["naz"])
daz = float(grid["daz"])

rg0 = float(grid["rg0"])
nrg = int(grid["nrg"])
drg = float(grid["drg"])

gedi_az = az0 + daz * np.arange(naz)
gedi_rg = rg0 + drg * np.arange(nrg)

# ---------------------------------------------------------
# 6. Verify alignment and dimensions
# ---------------------------------------------------------
assert cubeF.shape == cubeC.shape
assert cubeF.shape == (
    len(tomo_az),
    len(tomo_rg),
    len(zaxis),
)

assert gedi_height.shape == (len(gedi_az), len(gedi_rg))

print("TOMOSAR")
print(f"Cube shape: {cubeF.shape}")
print(f"Azimuth: {tomo_az[0]/1000:.3f}–{tomo_az[-1]/1000:.3f} km")
print(f"Range:   {tomo_rg[0]/1000:.3f}–{tomo_rg[-1]/1000:.3f} km")
print(f"Height:  {zaxis[0]:.1f}–{zaxis[-1]:.1f} m")

print("\nGEDI")
print(f"Map shape: {gedi_height.shape}")
print(f"Azimuth: {gedi_az[0]/1000:.3f}–{gedi_az[-1]/1000:.3f} km")
print(f"Range:   {gedi_rg[0]/1000:.3f}–{gedi_rg[-1]/1000:.3f} km")

print("\nSpatial coordinates restored successfully.")

### Plot

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

from datetime import datetime, timezone
from biomass_tomosar.paths import outputs_dir

# =========================================================
# 1. SETTINGS
# =========================================================
HEIGHT_MIN = 0              # Display height limits [m]
HEIGHT_MAX = 60

SURFACE_MAX_POINTS = 120    # Maximum samples per axis for 3D
SURFACE_ELEV = 28           # 3D camera elevation [degrees]
SURFACE_AZIM = -65          # 3D camera azimuth [degrees]

SAVE_FIGURES = True
FIGURE_DPI = 200

CMAP = "viridis"

# =========================================================
# 2. EXTRACT HEIGHT OF MAXIMUM RECONSTRUCTED POWER
# =========================================================
# cubeF, cubeC shape: (azimuth, range, height)
# Output shape: (azimuth, range)

def peak_height_map(cube, zaxis, chunk_size=32):
    """
    Find the height of maximum positive power at each pixel.

    Invalid profiles remain NaN.
    Process in small chunks to avoid large temporary arrays.
    """
    naz, nrg, nz = cube.shape

    assert nz == len(zaxis)

    height_map = np.full(
        (naz, nrg),
        np.nan,
        dtype=np.float32
    )

    for i in range(0, naz, chunk_size):
        j = min(i + chunk_size, naz)

        power = np.asarray(cube[i:j, :, :])

        valid = np.isfinite(power) & (power > 0)

        peak_idx = np.argmax(
            np.where(valid, power, -np.inf),
            axis=2
        )

        heights = zaxis[peak_idx].astype(np.float32)

        heights[~np.any(valid, axis=2)] = np.nan

        height_map[i:j] = heights

    return height_map


fourier_height_map = peak_height_map(cubeF, zaxis)
capon_height_map = peak_height_map(cubeC, zaxis)

# GEDI already provides forest height above ground
gedi_height_map = np.asarray(
    gedi_height,
    dtype=np.float32
).copy()

gedi_height_map[
    ~np.isfinite(gedi_height_map) |
    (gedi_height_map < 0)
] = np.nan

print("HEIGHT MAPS")
print("Fourier:", fourier_height_map.shape)
print("Capon:  ", capon_height_map.shape)
print("GEDI:   ", gedi_height_map.shape)

# =========================================================
# 3. PREPARE COMMON DISPLAY SETTINGS
# =========================================================
cmap = mpl.colormaps[CMAP].copy()
cmap.set_bad("lightgray")

norm = mpl.colors.Normalize(
    vmin=HEIGHT_MIN,
    vmax=HEIGHT_MAX
)

# Convert coordinates to km
tomo_az_km = np.asarray(tomo_az) / 1000
tomo_rg_km = np.asarray(tomo_rg) / 1000

gedi_az_km = np.asarray(gedi_az) / 1000
gedi_rg_km = np.asarray(gedi_rg) / 1000


def image_extent(az, rg):
    """Pixel-edge extent from pixel-center coordinates."""
    daz = np.median(np.diff(az))
    drg = np.median(np.diff(rg))

    return [
        rg[0] - drg / 2,
        rg[-1] + drg / 2,
        az[0] - daz / 2,
        az[-1] + daz / 2,
    ]


# =========================================================
# 4. FIGURE 1 — 2D HEIGHT MAPS
# =========================================================
fig_2d, axes = plt.subplots(
    1, 3,
    figsize=(19, 8),
    sharex=True,
    sharey=True,
    constrained_layout=True
)

maps = [
    ("Fourier — Peak Scattering Height",
     fourier_height_map, tomo_az_km, tomo_rg_km),

    ("Capon — Peak Scattering Height",
     capon_height_map, tomo_az_km, tomo_rg_km),

    ("GEDI — Canopy Height",
     gedi_height_map, gedi_az_km, gedi_rg_km),
]

for ax, (title, height_map, az, rg) in zip(axes, maps):

    im = ax.imshow(
        np.ma.masked_invalid(height_map),
        extent=image_extent(az, rg),
        origin="lower",
        aspect="equal",
        interpolation="nearest",
        cmap=cmap,
        norm=norm,
    )

    ax.set_title(title)
    ax.set_xlabel("Slant range [km]")

    # Identical map coverage for all panels
    ax.set_xlim(tomo_rg_km[0], tomo_rg_km[-1])
    ax.set_ylim(tomo_az_km[-1], tomo_az_km[0])

    ax.grid(alpha=0.1)

axes[0].set_ylabel("Azimuth [km]")

fig_2d.colorbar(
    im,
    ax=axes,
    label="Height [m]",
    shrink=0.7,
    extend="both",
)

fig_2d.suptitle(
    "BIOMASS TomoSAR — Fourier, Capon and GEDI Height Maps",
    fontsize=15,
)

# =========================================================
# 5. PREPARE DATA FOR 3D SURFACES
# =========================================================
# Use the same height maps, decimated for rendering only.
# The original 2D height maps remain at full resolution.

def prepare_surface(height_map, az, rg, max_points=120):
    step_az = max(1, int(np.ceil(len(az) / max_points)))
    step_rg = max(1, int(np.ceil(len(rg) / max_points)))

    az_plot = az[::step_az]
    rg_plot = rg[::step_rg]

    Z = height_map[::step_az, ::step_rg]

    X, Y = np.meshgrid(rg_plot, az_plot)

    return X, Y, np.ma.masked_invalid(Z)


surfaces = [
    prepare_surface(
        fourier_height_map,
        tomo_az_km,
        tomo_rg_km,
        SURFACE_MAX_POINTS
    ),
    prepare_surface(
        capon_height_map,
        tomo_az_km,
        tomo_rg_km,
        SURFACE_MAX_POINTS
    ),
    prepare_surface(
        gedi_height_map,
        gedi_az_km,
        gedi_rg_km,
        SURFACE_MAX_POINTS
    ),
]

# =========================================================
# 6. FIGURE 2 — 3D HEIGHT SURFACES
# =========================================================
fig_3d = plt.figure(
    figsize=(20, 8),
    constrained_layout=True
)

titles = [
    "Fourier — Peak Scattering Height",
    "Capon — Peak Scattering Height",
    "GEDI — Canopy Height",
]

axes_3d = []

for i, ((X, Y, Z), title) in enumerate(zip(surfaces, titles)):

    ax = fig_3d.add_subplot(
        1, 3, i + 1,
        projection="3d"
    )

    ax.plot_surface(
        X, Y, Z,
        cmap=cmap,
        norm=norm,
        linewidth=0,
        antialiased=False,
        rcount=X.shape[0],
        ccount=X.shape[1],
    )

    ax.set_title(title, pad=15)

    ax.set_xlabel("Slant range [km]", labelpad=8)
    ax.set_ylabel("Azimuth [km]", labelpad=8)
    ax.set_zlabel("Height [m]", labelpad=8)

    ax.set_xlim(tomo_rg_km[0], tomo_rg_km[-1])
    ax.set_ylim(tomo_az_km[-1], tomo_az_km[0])
    ax.set_zlim(HEIGHT_MIN, HEIGHT_MAX)

    ax.view_init(
        elev=SURFACE_ELEV,
        azim=SURFACE_AZIM
    )

    # Visual vertical exaggeration for clarity
    ax.set_box_aspect((1, 1.3, 0.6))

    axes_3d.append(ax)

sm = mpl.cm.ScalarMappable(
    norm=norm,
    cmap=cmap
)

sm.set_array([])

fig_3d.colorbar(
    sm,
    ax=axes_3d,
    label="Height [m]",
    shrink=0.55,
    pad=0.02,
    extend="both",
)

fig_3d.suptitle(
    "BIOMASS TomoSAR — 3D Height Surfaces",
    fontsize=15,
)

# =========================================================
# 7. SAVE BOTH FIGURES
# =========================================================
if SAVE_FIGURES:

    timestamp = datetime.now(timezone.utc).strftime(
        "%Y%m%d_%H%M%S_%f_UTC"
    )

    run_dir = (
        outputs_dir() / f"height_maps_{timestamp}"
    )

    run_dir.mkdir(
        parents=True,
        exist_ok=False
    )

    file_2d = run_dir / "height_maps_2d.png"
    file_3d = run_dir / "height_surfaces_3d.png"

    with file_2d.open("xb") as f:
        fig_2d.savefig(
            f,
            format="png",
            dpi=FIGURE_DPI,
            bbox_inches="tight"
        )

    with file_3d.open("xb") as f:
        fig_3d.savefig(
            f,
            format="png",
            dpi=FIGURE_DPI,
            bbox_inches="tight"
        )

    print("\nFigures saved to:")
    print(run_dir)

# =========================================================
# 8. DISPLAY
# =========================================================
plt.show()

### Analyze Coherence Matrix

## Troubleshooting and Diagnostic

### Compare Kz conventions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Use the pixel selected in the previous notebook cell
ia, ir = AZ_IDX, RG_IDX

# Extract the original multilooking window
y = xp_stack[
    :,
    az_starts[ia]:az_starts[ia] + looksa,
    rg_starts[ir]:rg_starts[ir] + looksr
].reshape(xp_stack.shape[0], -1)

valid = np.all(np.isfinite(y) & (y != 0), axis=0)
y = y[:, valid]

if y.shape[1] < 2:
    raise ValueError("Not enough valid looks")

# Sample covariance
R = y @ y.conj().T / y.shape[1]

# Local vertical wavenumbers
kkz = kz_ml[:, ia, ir]

if not np.all(np.isfinite(kkz)):
    raise ValueError("Invalid kz values")

# Symmetric height axis to reveal sign reversals
z_test = np.linspace(-120, 120, 481)

a_minus = np.exp(-1j * np.outer(kkz, z_test))
a_plus  = np.exp(+1j * np.outer(kkz, z_test))

def fourier_spectrum(R, a):
    return np.einsum(
        "iz,ij,jz->z",
        a.conj(), R, a,
        optimize=True
    ).real

# Real-data reconstructions
p_minus = fourier_spectrum(R, a_minus)
p_plus = fourier_spectrum(R, a_plus)

# Synthetic test: one scatterer at a known height
Z_TRUE = 35.0
y_synthetic = np.exp(-1j * kkz * Z_TRUE)
R_synthetic = np.outer(y_synthetic, y_synthetic.conj())

p_synthetic = fourier_spectrum(R_synthetic, a_minus)

# Convert to relative dB for this diagnostic
def relative_db(p):
    return 10 * np.log10(
        np.maximum(p / np.max(p), 1e-10)
    )

fig, axes = plt.subplots(
    1, 2,
    figsize=(14, 5),
    constrained_layout=True
)

# Synthetic validation
axes[0].plot(
    z_test, relative_db(p_synthetic),
    label="Reconstructed synthetic scatterer"
)
axes[0].axvline(
    Z_TRUE, color="red", linestyle="--",
    label=f"True height = {Z_TRUE:.1f} m"
)
axes[0].set_title("Synthetic point-scatterer test")

# Real data, compare phase signs
axes[1].plot(
    z_test, relative_db(p_minus),
    label="exp(-j kz z) — current"
)
axes[1].plot(
    z_test, relative_db(p_plus),
    label="exp(+j kz z) — alternative",
    linestyle="--"
)

if gedi_height_at_pixel is not None:
    axes[1].axvline(
        gedi_height_at_pixel,
        color="green", linestyle=":",
        label=f"GEDI = {gedi_height_at_pixel:.1f} m"
    )

axes[1].set_title("Real SLC — steering-vector sign test")

for ax in axes:
    ax.set_xlabel("Height [m]")
    ax.set_ylabel("Relative Fourier power [dB]")
    ax.set_ylim(-35, 2)
    ax.grid(alpha=0.3)
    ax.legend()

plt.show()

print(f"Synthetic true height: {Z_TRUE:.1f} m")
print(
    "Synthetic reconstructed peak:",
    f"{z_test[np.argmax(p_synthetic)]:.1f} m"
)

print(
    "Real peak, negative exponent:",
    f"{z_test[np.argmax(p_minus)]:.1f} m"
)
print(
    "Real peak, positive exponent:",
    f"{z_test[np.argmax(p_plus)]:.1f} m"
)

### Test Resolution

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =========================================================
# 1. SETTINGS
# =========================================================
GROUND_HEIGHT = 0.0
CANOPY_HEIGHT = 30.0

GROUND_POWER = 0.3
CANOPY_POWER = 1.0      # Try 0.25, 0.5, 1.0, 2.0

NOISE_POWER = 0.001
DIAGONAL_LOADING = 0.05

NORMALIZE_PROFILES = False

# Use previously selected TomoSAR pixel
ia, ir = AZ_IDX, RG_IDX

# Actual BIOMASS vertical wavenumbers at this pixel
kkz = kz_ml[:, ia, ir]

if not np.all(np.isfinite(kkz)):
    raise ValueError("Invalid kz values at selected pixel")

n_acq = len(kkz)

# Height axis
z_test = np.linspace(-40, 80, 481)

# =========================================================
# 2. SYNTHETIC SCATTERERS
# =========================================================
# Same steering-vector convention as our reconstruction
def steering(z):
    return np.exp(-1j * kkz * z)

a_ground = steering(GROUND_HEIGHT)
a_canopy = steering(CANOPY_HEIGHT)

# Covariance contribution of each scatterer
R_ground = (
    GROUND_POWER
    * np.outer(a_ground, a_ground.conj())
)

R_canopy = (
    CANOPY_POWER
    * np.outer(a_canopy, a_canopy.conj())
)

# Total covariance
R = (
    R_ground
    + R_canopy
    + NOISE_POWER * np.eye(n_acq)
)

# Steering matrix for every tested height
amat = np.exp(
    -1j * np.outer(kkz, z_test)
)

# =========================================================
# 3. FOURIER RECONSTRUCTION
# =========================================================
def fourier_spectrum(R, A):
    return np.einsum(
        "iz,ij,jz->z",
        A.conj(), R, A,
        optimize=True
    ).real

# Separate scatterer contributions
fourier_ground = fourier_spectrum(R_ground, amat)
fourier_canopy = fourier_spectrum(R_canopy, amat)

# Combined response
fourier_total = fourier_spectrum(R, amat)

# =========================================================
# 4. CAPON RECONSTRUCTION
# =========================================================
mean_power = np.trace(R).real / n_acq

R_loaded = R + (
    DIAGONAL_LOADING
    * mean_power
    * np.eye(n_acq)
)

# Solve R_loaded X = A
X = np.linalg.solve(R_loaded, amat)

denominator = np.sum(
    amat.conj() * X,
    axis=0
).real

capon_total = 1.0 / denominator

# =========================================================
# 5. CONVERT TO dB
# =========================================================
def to_db(power, normalize=False):
    power = np.maximum(np.asarray(power), 1e-12)

    if normalize:
        power = power / np.max(power)

    return 10 * np.log10(power)

F_total_db = to_db(fourier_total, NORMALIZE_PROFILES)
F_ground_db = to_db(fourier_ground, NORMALIZE_PROFILES)
F_canopy_db = to_db(fourier_canopy, NORMALIZE_PROFILES)

C_total_db = to_db(capon_total, NORMALIZE_PROFILES)

# =========================================================
# 6. PLOT
# =========================================================
fig, axes = plt.subplots(
    2, 1,
    figsize=(12, 9),
    sharex=True,
    constrained_layout=True
)

# Fourier
axes[0].plot(
    z_test, F_total_db,
    linewidth=2.5,
    label="Combined Fourier"
)

axes[0].plot(
    z_test, F_ground_db,
    "--",
    alpha=0.7,
    label="Ground contribution"
)

axes[0].plot(
    z_test, F_canopy_db,
    "--",
    alpha=0.7,
    label="Canopy contribution"
)

axes[0].set_title("Fourier — Two Point Scatterers")

# Capon
axes[1].plot(
    z_test, C_total_db,
    linewidth=2.5,
    label="Combined Capon"
)

axes[1].set_title("Capon — Two Point Scatterers")
axes[1].set_xlabel("Height [m]")

for ax in axes:

    ax.axvline(
        GROUND_HEIGHT,
        color="black",
        linestyle=":",
        linewidth=1.5,
        label="True ground (0 m)"
    )

    ax.axvline(
        CANOPY_HEIGHT,
        color="green",
        linestyle=":",
        linewidth=1.5,
        label="True canopy (30 m)"
    )

    ax.set_ylabel(
        "Relative power [dB]"
        if NORMALIZE_PROFILES
        else "Reconstructed power [dB]"
    )

    ax.grid(alpha=0.3)
    ax.legend()

plt.show()

# =========================================================
# 7. PEAK LOCATIONS
# =========================================================
z_fourier_peak = z_test[np.argmax(fourier_total)]
z_capon_peak = z_test[np.argmax(capon_total)]

print("TWO-SCATTERER SIMULATION")
print(f"Ground height:       {GROUND_HEIGHT:.1f} m")
print(f"Canopy height:       {CANOPY_HEIGHT:.1f} m")
print(f"Ground power:        {GROUND_POWER:.2f}")
print(f"Canopy power:        {CANOPY_POWER:.2f}")

print(f"\nFourier maximum:     {z_fourier_peak:.2f} m")
print(f"Capon maximum:       {z_capon_peak:.2f} m")

print(
    f"\nNominal resolution:  "
    f"{2*np.pi/np.ptp(kkz):.2f} m"
)